In [1]:
# ------------------------------------------------------------------------------------------------------
# task_1_dataloader.py imports
import glob
import json
import os
import re
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
#from Post_Embedder import PostEmbedder


#-------------------------------------------------------------------------------------------------------
# Post_Embedder.py imports
import re
import numpy as np
import spacy
import gensim
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer



#-------------------------------------------------------------------------------------------------------
# topk_similar_dataset.py imports 
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Import your existing types – adjust the import name if needed
#from task_1_dataloader import Post, Timeline, SelfState


#-------------------------------------------------------------------------------------------------------
# train.py imports
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "4"
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json
import torch
import torch.nn as nn
import numpy as np
import pickle # Added for caching
from torch.utils.data import DataLoader
from transformers import Qwen2Model, Qwen2PreTrainedModel, AutoTokenizer , BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# Import your existing dataloader and dataset classes
#from task_1_dataloader import load_all_timelines
#from topk_similar_dataset import PostIndex, TopKSimilarDataset






/home/mudasir/miniconda3/envs/clpsych/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Post_Embedder.py
# ── Twitter-RoBERTa task names ───────────────────────────────────────────────
_TASKS = ["emoji", "emotion", "hate", "irony", "offensive", "sentiment"]


class PostEmbedder:
    def __init__(self, wv_model_path: str, spacy_model: str = "en_core_web_sm", device: str = "cpu") -> None:
        print("[PostEmbedder] Loading Word2Vec …")
        self.wv_model = gensim.models.KeyedVectors.load_word2vec_format( wv_model_path, binary=False)
        self._wdim = self.wv_model["word"].shape[0]

        print("[PostEmbedder] Loading sentence-transformer …")
        self.sv_model = SentenceTransformer("sentence-transformers/nli-roberta-large", device=device)

        print("[PostEmbedder] Loading Twitter-RoBERTa task models …")
        self._task_models: dict[str, tuple] = {}
        for task in _TASKS:
            model_name = (f"cardiffnlp/twitter-roberta-base-{task}-latest" if task == "hate" else f"cardiffnlp/twitter-roberta-base-{task}")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
            model.eval()
            self._task_models[task] = (model, tokenizer)

        print("[PostEmbedder] Loading spaCy …")
        self.nlp = spacy.load(spacy_model)

        self._stops = set(stopwords.words("english"))
        self._device = device
        print("[PostEmbedder] Ready.")


    def embed(self, text: str) -> np.ndarray:
        try:
            sentences = [str(s) for s in self.nlp(text).sents]
            if not sentences:
                sentences = [text]

            wv_part = self._word2vec_emb(text)
            sv_part = self._sentence_emb(sentences)
            task_part = self._task_scores(sentences)

            vec = np.concatenate([wv_part, sv_part, task_part], axis=None)

            if np.isnan(vec).any():
                raise ValueError(f"NaN values in embedding for text: {text[:60]!r}")
            return vec
        except Exception as exc:
            raise RuntimeError(f"[PostEmbedder] embed() failed: {exc}") from exc


    @staticmethod
    def _preprocess(text: str) -> str:
        """Lower-case; replace @mentions with @user; strip URLs."""
        tokens = []
        for t in text.split():
            t = t.lower()
            if t.startswith("@") and len(t) > 1:
                t = "@user"
            elif t.startswith("http"):
                t = ""
            tokens.append(t)
        return " ".join(tokens)

    def _remove_stopwords(self, text: str) -> list[str]:
        return [w for w in text.split() if w and w not in self._stops]

    def _word2vec_emb(self, text: str) -> np.ndarray:
        cleaned = self._preprocess(text)
        words = self._remove_stopwords(cleaned)
        vec = np.zeros(self._wdim)
        n = 0
        for w in words:
            if w in self.wv_model:
                vec += self.wv_model[w]
                n += 1
        if n > 0:
            vec /= n
        return vec

    def _sentence_emb(self, sentences: list[str]) -> np.ndarray:
        embeddings = self.sv_model.encode(sentences, device=self._device)
        return np.mean(embeddings, axis=0)

    def _task_score_single(self, task: str, text: str) -> np.ndarray:
        model, tokenizer = self._task_models[task]
        enc = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(self._device)
        with torch.no_grad():
            out = model(**enc)
        return out[0][0].detach().cpu().numpy()

    def _task_scores(self, sentences: list[str]) -> np.ndarray:
        """Average Twitter-RoBERTa scores across all sentences (hate excluded from concat)."""
        per_sentence = []
        for sent in sentences:
            parts = [
                self._task_score_single(t, sent)
                for t in ["emoji", "emotion", "irony", "offensive", "sentiment" , "hate"]
            ]
            per_sentence.append(np.concatenate(parts, axis=None))
        return np.mean(per_sentence, axis=0)

In [3]:

WV_MODEL_PATH  = "./Task_1/wiki-news-300d-1M.vec"
device = "cuda" if torch.cuda.is_available() else "cpu"
PostEmbedder = PostEmbedder(wv_model_path=WV_MODEL_PATH, device=device)

ABCD_TAXONOMY = {
    "A": {
        "adaptive": {
            1:  "Calm/laid back",
            3:  "Sad, Emotional pain, grieving",
            5:  "Content, happy, joy, hopeful",
            7:  "Vigor/energetic",
            9:  "Justifiable anger/assertive anger, justifiable outrage",
            11: "Proud",
            13: "Feel loved, belong",
        },
        "maladaptive": {
            2:  "Anxious/fearful/tense",
            4:  "Depressed, despair, hopeless",
            6:  "Mania",
            8:  "Apathetic, don't care, blunted",
            10: "Angry (aggression), disgust, contempt",
            12: "Ashamed, guilty",
            14: "Feel lonely",
        },
    },
    "B-O": {
        "adaptive": {
            1: "Relating behavior",
            3: "Autonomous or adaptive control behavior",
        },
        "maladaptive": {
            2: "Fight or flight behavior",
            4: "Over controlled or controlling behavior",
        },
    },
    "B-S": {
        "adaptive": {
            1: "Self care and improvement",
        },
        "maladaptive": {
            2: "Self harm, neglect and avoidance",
        },
    },
    "C-O": {
        "adaptive": {
            1: "Perception of the other as related",
            3: "Perception of the other as facilitating autonomy needs",
        },
        "maladaptive": {
            2: "Perception of the other as detached or over attached",
            4: "Perception of the other as blocking autonomy needs",
        },
    },
    "C-S": {
        "adaptive": {
            1: "Self-acceptance and compassion",
        },
        "maladaptive": {
            2: "Self criticism",
        },
    },
    "D": {
        "adaptive": {
            1: "Relatedness",
            3: "Autonomy and adaptive control",
            5: "Competence, self esteem, self-care",
        },
        "maladaptive": {
            2: "Expectation that relatedness needs will not be met",
            4: "Expectation that autonomy needs will not be met",
            6: "Expectation that competence needs will not be met",
        },
    },
}


# Canonical dimension keys (as they appear in JSON)
DIMENSIONS = ["A", "B-O", "B-S", "C-O", "C-S", "D"]

# Change label constants
NO_CHANGE  = "0"
SWITCH     = "S"
ESCALATION = "E"

# Presence scale
PRESENCE_MIN = 1
PRESENCE_MAX = 5

# Date format in JSON
DATE_FORMAT = "%d-%m-%Y, %H:%M:%S"

@dataclass
class SubElement:
    dimension: str        # "A", "B-O", "B-S", "C-O", "C-S", "D"
    valence:   str        # "adaptive" | "maladaptive"
    number:    int        # e.g. 4 for "(4) Depressed, despair, hopeless"
    label:     str        # e.g. "Depressed, despair, hopeless"
    span:      str        # highlighted_evidence from post text

    @classmethod
    def from_json(cls, dimension: str, value: Dict, valence: str) -> "SubElement":
        cat_raw = value.get("Category", "")
        match = re.match(r"\((\d+)\)\s*(.+)", cat_raw)
        if match:
            number = int(match.group(1))
            label  = match.group(2).strip()
        else:
            number = 0
            label  = cat_raw.strip()

        # Fill label from taxonomy if blank
        tax_labels = ABCD_TAXONOMY.get(dimension, {}).get(valence, {})
        if number in tax_labels and not label:
            label = tax_labels[number]

        return cls(
            dimension=dimension,
            valence=valence,
            number=number,
            label=label,
            span=value.get("highlighted_evidence", "").strip(),
        )

    @property
    def short_tag(self) -> str:
        """e.g. 'A-(4)' - used in prompts and summaries."""
        return f"{self.dimension}-({self.number})"

    @property
    def full_tag(self) -> str:
        """e.g. 'A - (4) Depressed, despair, hopeless'"""
        return f"{self.dimension} - ({self.number}) {self.label}"


@dataclass
class SelfState:
    valence:     str               # "adaptive" | "maladaptive"
    subelements: List[SubElement]  # one per dimension at most (Task 1.1)
    presence:    int               # 1-5 (Task 1.2); 1 = not present

    @property
    def by_dimension(self) -> Dict[str, SubElement]:
        return {se.dimension: se for se in self.subelements}

    @property
    def dimensions_present(self) -> List[str]:
        return [se.dimension for se in self.subelements]

    @property
    def is_present(self) -> bool:
        """A self-state is considered present if presence > 1."""
        return self.presence > 1

    def to_prompt_dict(self) -> Dict:
        """Serialise back to the same JSON evidence format for prompting."""
        d = {}
        for se in self.subelements:
            d[se.dimension] = {
                "Category": f"({se.number}) {se.label}",
                "highlighted_evidence": se.span,
            }
        d["Presence"] = self.presence
        return d

def _parse_self_state(block: Dict, valence: str) -> SelfState:
    subelements = []
    presence = 1  # default: not present

    for key, value in block.items():
        if key == "Presence":
            try:
                presence = max(PRESENCE_MIN, min(PRESENCE_MAX, int(value)))
            except (TypeError, ValueError):
                presence = 1
            continue
        if not isinstance(value, dict):
            continue
        try:
            se = SubElement.from_json(key, value, valence)
            subelements.append(se)
        except Exception:
            pass

    # If no subelements found, presence must be 1 (per task spec)
    if not subelements:
        presence = 1

    return SelfState(valence=valence, subelements=subelements, presence=presence)


@dataclass
class Post:
    post_id:    str
    post_index: int
    text:       str
    timestamp:  datetime

    # Task 2: Change labels (INDEPENDENT - both can be set simultaneously)
    switch_label:     str  # "S" | "0"
    escalation_label: str  # "E" | "0"

    # Task 1.2: Well-being score (GAF-based, 1-10 or None)
    wellbeing: Optional[int]

    # Task 1.1 + 1.2: Gold self-states
    adaptive_state:    SelfState  # valence="adaptive"
    maladaptive_state: SelfState  # valence="maladaptive"

    # Whether this post has any annotation
    is_annotated: bool = False

    # Predictions (filled by pipeline)
    pred_adaptive_state:    Optional[SelfState] = None
    pred_maladaptive_state: Optional[SelfState] = None
    pred_switch_label:     str = "0"
    pred_escalation_label: str = "0"
    temporal_embedding: Optional[np.ndarray] = None
    post_embedding: Optional[np.ndarray] = None

    @property
    def is_switch(self) -> bool:
        return self.switch_label == SWITCH

    @property
    def is_escalation(self) -> bool:
        return self.escalation_label == ESCALATION

    @property
    def has_change(self) -> bool:
        return self.is_switch or self.is_escalation

    @property
    def change_tag(self) -> str:
        """Human-readable tag: 'S', 'E', 'S+E', or '-'."""
        tags = []
        if self.is_switch:     tags.append("S")
        if self.is_escalation: tags.append("E")
        return "+".join(tags) if tags else "-"

    @property
    def adaptive_presence(self) -> int:
        return self.adaptive_state.presence

    @property
    def maladaptive_presence(self) -> int:
        return self.maladaptive_state.presence

    @classmethod
    def from_dict(cls, d: Dict) -> "Post":
        try:
            ts = datetime.strptime(d["date"], DATE_FORMAT)
        except (ValueError, KeyError):
            ts = datetime.min

        switch_label     = SWITCH     if str(d.get("Switch",     "0")).upper() == "S" else "0"
        escalation_label = ESCALATION if str(d.get("Escalation", "0")).upper() == "E" else "0"

        wb = d.get("Well-being")
        wellbeing = int(wb) if wb is not None else None

        evidence = d.get("evidence", {})
        adaptive_state    = _parse_self_state(evidence.get("adaptive-state",    {}), "adaptive")
        maladaptive_state = _parse_self_state(evidence.get("maladaptive-state", {}), "maladaptive")

        is_annotated = (
            bool(adaptive_state.subelements)
            or bool(maladaptive_state.subelements)
            or wellbeing is not None
        )

        post_embedding = PostEmbedder.embed(d.get("post", ""))

        return cls(
            post_id=d.get("post_id", ""),
            post_index=int(d.get("post_index", 0)),
            text=d.get("post", ""),
            timestamp=ts,
            switch_label=switch_label,
            escalation_label=escalation_label,
            wellbeing=wellbeing,
            adaptive_state=adaptive_state,
            maladaptive_state=maladaptive_state,
            is_annotated=is_annotated,
            post_embedding=post_embedding,
        )

@dataclass
class Timeline:
    """A complete, chronologically ordered sequence of posts for one user."""
    timeline_id: str
    posts: List[Post]

    # Stats (computed on init)
    n_posts:      int = 0
    n_annotated:  int = 0
    n_switches:   int = 0
    n_escalations: int = 0

    def __post_init__(self):
        self.posts.sort(key=lambda p: (p.timestamp, p.post_index))
        self.n_posts       = len(self.posts)
        self.n_annotated   = sum(1 for p in self.posts if p.is_annotated)
        self.n_switches    = sum(1 for p in self.posts if p.is_switch)
        self.n_escalations = sum(1 for p in self.posts if p.is_escalation)

    def hours_between(self, idx_a: int, idx_b: int) -> float:
        delta = self.posts[idx_b].timestamp - self.posts[idx_a].timestamp
        return max(0.0, delta.total_seconds() / 3600)

    def get_context(self, post_idx: int, window: int = 5) -> List[Post]:
        """Return up to `window` posts BEFORE post_idx (exclusive)."""
        start = max(0, post_idx - window)
        return self.posts[start:post_idx]

    @classmethod
    def from_dict(cls, d: Dict) -> "Timeline":
        posts = [Post.from_dict(p) for p in d.get("posts", [])]
        return cls(timeline_id=d.get("timeline_id", ""), posts=posts)



def load_timeline_file(path: str) -> Timeline:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return Timeline.from_dict(data)


def load_all_timelines(data_dir: str, pattern: str = "*.json") -> List[Timeline]:
    paths = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not paths:
        raise FileNotFoundError(f"No '{pattern}' files found in: {data_dir}")

    timelines = []
    for path in paths:
        try:
            timelines.append(load_timeline_file(path))
        except Exception as e:
            print(f"[WARNING] Skipping {path}: {e}")

    print(f"\nLoaded {len(timelines)} timelines from: {data_dir}")
    _print_dataset_stats(timelines)
    return timelines


def _print_dataset_stats(timelines: List[Timeline]) -> None:
    total_posts  = sum(tl.n_posts for tl in timelines)
    total_ann    = sum(tl.n_annotated for tl in timelines)
    total_sw     = sum(tl.n_switches for tl in timelines)
    total_esc    = sum(tl.n_escalations for tl in timelines)
    both         = sum(1 for tl in timelines
                       for p in tl.posts if p.is_switch and p.is_escalation)
    ada_subs     = sum(len(p.adaptive_state.subelements)
                       for tl in timelines for p in tl.posts)
    mal_subs     = sum(len(p.maladaptive_state.subelements)
                       for tl in timelines for p in tl.posts)

    print(f"  Timelines             : {len(timelines)}")
    print(f"  Total posts           : {total_posts}")
    print(f"  Annotated posts       : {total_ann}")
    print(f"  Switch posts          : {total_sw}")
    print(f"  Escalation posts      : {total_esc}")
    print(f"  Both (S+E) posts      : {both}")
    print(f"  Adaptive subelements  : {ada_subs}")
    print(f"  Maladaptive subelements: {mal_subs}")



def _empty_self_state(valence: str) -> SelfState:
    return SelfState(valence=valence, subelements=[], presence=1)


def load_test_timelines(test_dir: str, embedder: PostEmbedder) -> list[Timeline]:
    paths     = sorted(glob.glob(os.path.join(test_dir, "*.json")))
    timelines = []

    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        posts = []
        for d in data.get("posts", []):
            text = d.get("post", "")
            emb  = embedder.embed(text)

            # Parse timestamp
            try:
                ts = datetime.strptime(d["date"], DATE_FORMAT)
            except Exception:
                ts = datetime.min

            post = Post(
                post_id          = d.get("post_id", ""),
                post_index       = int(d.get("post_index", 0)),
                text             = text,
                timestamp        = ts,
                switch_label     = "0",
                escalation_label = "0",
                wellbeing        = None,
                adaptive_state   = _empty_self_state("adaptive"),
                maladaptive_state= _empty_self_state("maladaptive"),
                is_annotated     = False,
                post_embedding   = emb,
            )
            posts.append(post)

        timelines.append(Timeline(
            timeline_id=data.get("timeline_id", ""),
            posts=posts,
        ))

    print(f"Loaded {len(timelines)} test timelines  "
          f"({sum(len(t.posts) for t in timelines)} posts)")
    return timelines


[PostEmbedder] Loading Word2Vec …
[PostEmbedder] Loading sentence-transformer …


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11036.08it/s]
RobertaModel LOAD REPORT from: sentence-transformers/nli-roberta-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PostEmbedder] Loading Twitter-RoBERTa task models …


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 38681.12it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emoji
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 37552.57it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-emotion
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 48111.35it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/

[PostEmbedder] Loading spaCy …
[PostEmbedder] Ready.


In [4]:
#topk_similar_dataset.py
class PostIndex:
    def __init__(self, timelines: List[Timeline], exclude_same_timeline: bool = True, skip_no_embedding: bool = True) -> None:
        self.exclude_same_timeline = exclude_same_timeline

        # Collect all (timeline_id, Post) pairs that have a valid embedding
        self._entries: List[Tuple[str, Post]] = []

        for tl in timelines:
            for post in tl.posts:
                if post.post_embedding is None:
                    if not skip_no_embedding:
                        raise ValueError(
                            f"Post {post.post_id!r} has no embedding. "
                            "Run PostEmbedder first."
                        )
                    continue
                self._entries.append((tl.timeline_id, post))

        if not self._entries:
            raise ValueError("No posts with embeddings found in the provided timelines.")

        # Stack into (N, D) float32 matrix and L2-normalise rows for cosine sim
        raw = np.stack([e[1].post_embedding for e in self._entries], axis=0).astype(np.float32)
        norms = np.linalg.norm(raw, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)   # avoid /0 for zero vectors
        self._matrix = raw / norms                  # shape (N, D), unit vectors

        self._timeline_ids = [e[0] for e in self._entries]
        self._posts        = [e[1] for e in self._entries]

        print( f"[PostIndex] Built index: {len(self._entries)} posts, "f"embedding dim={raw.shape[1]}")

    def query( self, post: Post, query_timeline_id: str, k: int) -> Tuple[List[Post], List[float]]:
        if post.post_embedding is None:
            return [], []

        # Normalise query vector
        q = post.post_embedding.astype(np.float32)
        q_norm = np.linalg.norm(q)
        if q_norm > 0:
            q = q / q_norm

        # Cosine similarities: dot(matrix, q) because rows are already normalised
        sims = self._matrix @ q  # shape (N,)

        # Mask out: (a) the query post itself, (b) same-timeline posts if requested
        for i, (tid, p) in enumerate(self._entries):
            if p.post_id == post.post_id:
                sims[i] = -2.0   # guaranteed lowest
            elif self.exclude_same_timeline and tid == query_timeline_id:
                sims[i] = -2.0

        # Top-k indices (descending)
        top_k_idx = np.argpartition(sims, -k)[-k:]          # unsorted
        top_k_idx = top_k_idx[np.argsort(sims[top_k_idx])[::-1]]  # sorted desc

        similar_posts = [self._posts[i] for i in top_k_idx]
        scores        = [float(sims[i])  for i in top_k_idx]

        return similar_posts, scores

@dataclass
class TopKSimilarInstance:
    timeline_id:   str
    post:          Post
    similar_posts: List[Post]        # length == k (or fewer at dataset edges)
    scores:        List[float]       # parallel to similar_posts

    # ── convenience pass-throughs so code that reads Task11Instance still works
    @property
    def post_id(self)        -> str:           return self.post.post_id
    @property
    def post_index(self)     -> int:           return self.post.post_index
    @property
    def text(self)           -> str:           return self.post.text
    @property
    def adaptive_state(self) -> SelfState:     return self.post.adaptive_state
    @property
    def maladaptive_state(self) -> SelfState:  return self.post.maladaptive_state
    @property
    def wellbeing(self)      -> Optional[int]: return self.post.wellbeing
    @property
    def post_embedding(self) -> Optional[np.ndarray]: return self.post.post_embedding

    def similar_texts(self) -> List[str]:
        """Convenience: just the text of each similar post."""
        return [p.text for p in self.similar_posts]


# ── 3. Dataset ────────────────────────────────────────────────────────────────

class TopKSimilarDataset(Dataset):
    def __init__(self, timelines: List[Timeline], index: PostIndex, k: int = 5, annotated_only: bool = True) -> None:
        self.k = k
        self.instances: List[TopKSimilarInstance] = []

        skipped = 0
        for tl in timelines:
            for post in tl.posts:
                if annotated_only and not post.is_annotated:
                    continue
                if post.post_embedding is None:
                    skipped += 1
                    continue

                similar_posts, scores = index.query(post, tl.timeline_id, k=k)

                self.instances.append(TopKSimilarInstance(
                    timeline_id=tl.timeline_id,
                    post=post,
                    similar_posts=similar_posts,
                    scores=scores,
                ))

        print(
            f"[TopKSimilarDataset] {len(self.instances)} instances built "
            f"(k={k}, skipped {skipped} posts without embedding)"
        )

    def __len__(self) -> int:
        return len(self.instances)

    def __getitem__(self, i: int) -> TopKSimilarInstance:
        return self.instances[i]

    # ── collate ───────────────────────────────────────────────────────────────

    @staticmethod
    def collate(batch: List[TopKSimilarInstance]) -> dict:
        posts         = [inst.post          for inst in batch]
        similar_posts = [inst.similar_posts for inst in batch]
        scores        = torch.tensor([inst.scores for inst in batch], dtype=torch.float32)

        ada_presence = torch.tensor(
            [inst.post.adaptive_state.presence    for inst in batch],
            dtype=torch.float32,
        )
        mal_presence = torch.tensor(
            [inst.post.maladaptive_state.presence for inst in batch],
            dtype=torch.float32,
        )

        # Stack current-post embeddings if available
        embs = [inst.post.post_embedding for inst in batch]
        if all(e is not None for e in embs):
            post_embeddings = torch.tensor(np.stack(embs), dtype=torch.float32)
        else:
            post_embeddings = None

        # Stack similar-post embeddings: shape (B, k, D)
        sim_embs = [[p.post_embedding for p in inst.similar_posts] for inst in batch]
        if all(e is not None for row in sim_embs for e in row):
            similar_embeddings = torch.tensor(
                np.stack([np.stack(row) for row in sim_embs]), dtype=torch.float32
            )
        else:
            similar_embeddings = None

        return {
            "posts":              posts,
            "similar_posts":      similar_posts,
            "scores":             scores,             # (B, k)
            "post_embeddings":    post_embeddings,    # (B, D)
            "similar_embeddings": similar_embeddings, # (B, k, D)
            "ada_presence":       ada_presence,       # (B,)
            "mal_presence":       mal_presence,       # (B,)
        }

In [5]:
#model.py

TAXONOMY_TO_INDEX = {
    "adaptive": {
        "A":   {1: 0, 3: 1, 5: 2, 7: 3, 9: 4, 11: 5, 13: 6},
        "B-O": {1: 7, 3: 8},
        "B-S": {1: 9},
        "C-O": {1: 10, 3: 11},
        "C-S": {1: 12},
        "D":   {1: 13, 3: 14, 5: 15},
    },
    "maladaptive": {
        "A":   {2: 16, 4: 17, 6: 18, 8: 19, 10: 20, 12: 21, 14: 22},
        "B-O": {2: 23, 4: 24},
        "B-S": {2: 25},
        "C-O": {2: 26, 4: 27},
        "C-S": {2: 28},
        "D":   {2: 29, 4: 30, 6: 31},
    }
}

INDEX_TO_TAXONOMY = {}
for valence, elements in TAXONOMY_TO_INDEX.items():
    for element, subelements in elements.items():
        for number, index in subelements.items():
            INDEX_TO_TAXONOMY[index] = {"valence": valence, "element": element, "number": number}

ELEMENT_SLICES = {
    "adaptive": {
        "A": (0, 7), "B-O": (7, 9), "B-S": (9, 10), 
        "C-O": (10, 12), "C-S": (12, 13), "D": (13, 16)
    },
    "maladaptive": {
        "A": (16, 23), "B-O": (23, 25), "B-S": (25, 26), 
        "C-O": (26, 28), "C-S": (28, 29), "D": (29, 32)
    }
}



class QwenSelfStatePredictor(Qwen2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.model = Qwen2Model(config)
        self.num_subelements = 32
        self.num_presence = 2
        self.classifier = nn.Linear(config.hidden_size, self.num_subelements + self.num_presence)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        
        # Get hidden state of the last non-padded token
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = input_ids.shape[0]
        last_hidden_states = outputs.last_hidden_state[torch.arange(batch_size, device=input_ids.device), sequence_lengths]
        
        logits = self.classifier(last_hidden_states)
        subelement_logits = logits[:, :self.num_subelements]
        presence_preds = logits[:, self.num_subelements:]
        
        # Loss calculation removed from forward pass
        return {"subelement_logits": subelement_logits, "presence_preds": presence_preds}

def decode_predictions(subelement_logits, presence_preds, threshold=0.5):
    """Translates tensors to the JSON dictionary format expected by CLPsych."""
    probs = torch.sigmoid(subelement_logits)
    
    ada_presence = max(1, min(5, round(presence_preds[0].item())))
    mal_presence = max(1, min(5, round(presence_preds[1].item())))
    
    prediction = {
        "adaptive-state": {"Presence": ada_presence},
        "maladaptive-state": {"Presence": mal_presence}
    }
    
    for valence in ["adaptive", "maladaptive"]:
        state_key = f"{valence}-state"
        
        for element, (start_idx, end_idx) in ELEMENT_SLICES[valence].items():
            element_probs = probs[start_idx:end_idx]
            max_prob, max_local_idx = torch.max(element_probs, dim=0)
            
            if max_prob.item() >= threshold:
                global_idx = start_idx + max_local_idx.item()
                predicted_number = INDEX_TO_TAXONOMY[global_idx]["number"]
                prediction[state_key][element] = {"subelement": predicted_number}

        # Spec constraint: if no subelements, presence MUST be 1
        if len(prediction[state_key]) == 1: # Only 'Presence' key exists
            prediction[state_key]["Presence"] = 1

    return prediction

In [ ]:
# train.py


def vectorize_target(adaptive_state, maladaptive_state):
    subelements_vec = torch.zeros(32, dtype=torch.float32)
    presence_vec = torch.zeros(2, dtype=torch.float32)
    
    if adaptive_state and hasattr(adaptive_state, 'subelements'):
        for se in adaptive_state.subelements:
            idx = TAXONOMY_TO_INDEX["adaptive"].get(se.dimension, {}).get(se.number)
            if idx is not None:
                subelements_vec[idx] = 1.0
                
    if maladaptive_state and hasattr(maladaptive_state, 'subelements'):
        for se in maladaptive_state.subelements:
            idx = TAXONOMY_TO_INDEX["maladaptive"].get(se.dimension, {}).get(se.number)
            if idx is not None:
                subelements_vec[idx] = 1.0

    presence_vec[0] = float(adaptive_state.presence) if adaptive_state else 1.0
    presence_vec[1] = float(maladaptive_state.presence) if maladaptive_state else 1.0
    
    return subelements_vec, presence_vec

def qwen_custom_collate(batch):
    batch_prompts = []
    batch_subelements = []
    batch_presence = []
    raw_posts = []

    for inst in batch:
        raw_posts.append(inst.post)
        lines = []
        # Construct Few-Shot Context
        for rank, (ctx_post, score) in enumerate(zip(inst.similar_posts, inst.scores), 1):
            lines.append(f"### Example {rank}  (similarity: {score:.3f})")
            lines.append(f'Post: "{ctx_post.text}"')
            lines.append("Output:")
            
            lines.append("  Adaptive Self-State:")
            if ctx_post.adaptive_state.subelements:
                for se in ctx_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {ctx_post.adaptive_state.presence} / 5")
            
            lines.append("  Maladaptive Self-State:")
            if ctx_post.maladaptive_state.subelements:
                for se in ctx_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {ctx_post.maladaptive_state.presence} / 5\n")

        # Current Query Post
        lines.append("### Current Post")
        lines.append(f'Post: "{inst.text}"')
        lines.append("Output:")
        
        batch_prompts.append("\n".join(lines))

        # Vectorize Targets
        sub_vec, pres_vec = vectorize_target(inst.post.adaptive_state, inst.post.maladaptive_state)
        batch_subelements.append(sub_vec)
        batch_presence.append(pres_vec)

    return {
        "prompts": batch_prompts,
        "labels_subelements": torch.stack(batch_subelements),
        "labels_presence": torch.stack(batch_presence),
        "raw_posts": raw_posts,
        "timeline_ids": [inst.timeline_id for inst in batch]
    }






device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Hyperparameters ---
MODEL_NAME = "Qwen/Qwen2.5-7B"#"Qwen/Qwen2-1.5B" 
DATA_DIR = "./data/train"      
EVAL_DIR = "./data/val"        
CACHE_DIR = "./Task_1/dataset_cache"     # Directory to store cached datasets
BATCH_SIZE = 1
EPOCHS = 1
LR = 5e-5
K = 5 
SAVE_DIR = f".Task_1/saved_qwen_clpsych/{MODEL_NAME}_epoch{EPOCHS}" 

os.makedirs(CACHE_DIR, exist_ok=True)
train_cache_path = os.path.join(CACHE_DIR, f"train_dataset_k{K}.pkl")
eval_cache_path = os.path.join(CACHE_DIR, f"eval_dataset_k{K}.pkl")

# --- 1. Load Data (with Caching) ---
if os.path.exists(train_cache_path) and os.path.exists(eval_cache_path):
    print("Loading previously cached datasets from disk...")
    with open(train_cache_path, "rb") as f:
        train_dataset = pickle.load(f)
    with open(eval_cache_path, "rb") as f:
        eval_dataset = pickle.load(f)
    print("Cached datasets loaded successfully!")
else:
    print("Loading timelines and building indices (this will take a while but will be cached)...")
    train_timelines = load_all_timelines(DATA_DIR)
    eval_timelines = load_all_timelines(EVAL_DIR)
    
    train_index = PostIndex(train_timelines, exclude_same_timeline=True)
    
    print("Building Top-K datasets...")
    train_dataset = TopKSimilarDataset(train_timelines, train_index, k=K, annotated_only=True)
    eval_dataset = TopKSimilarDataset(eval_timelines, train_index, k=K, annotated_only=True) 
    
    print(f"Saving datasets to cache directory '{CACHE_DIR}' for faster future runs...")
    with open(train_cache_path, "wb") as f:
        pickle.dump(train_dataset, f)
    with open(eval_cache_path, "wb") as f:
        pickle.dump(eval_dataset, f)
    print("Cache saved!")


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=qwen_custom_collate)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=qwen_custom_collate)

# --- 2. Load Tokenizer & Model ---
print("Initializing Qwen Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token 

model = QwenSelfStatePredictor.from_pretrained(MODEL_NAME , quantization_config =BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier"]
) , device_map="auto")

# Setup LoRA (PEFT)
peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["classifier"],
    lora_dropout=0.05,
)
model = get_peft_model(model, peft_config)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# Define loss functions for the training loop
bce_loss_fn = nn.BCEWithLogitsLoss()
mse_loss_fn = nn.MSELoss()

# --- 3. Training Loop ---
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        inputs = tokenizer(batch["prompts"], padding=True, truncation=True, max_length=1536, return_tensors="pt").to(device)
        labels_subelements = batch["labels_subelements"].to(device)
        labels_presence = batch["labels_presence"].to(device)
        
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )
            
            subelement_logits = outputs["subelement_logits"]
            presence_preds = outputs["presence_preds"]
            
            loss_subelements = bce_loss_fn(subelement_logits.float(), labels_subelements)
            loss_presence = mse_loss_fn(presence_preds.float(), labels_presence)
            
            # Weighting factor
            loss = loss_subelements + (0.5 * loss_presence)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        if step % 10 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step} | Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    print(f"--- Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f} ---")

# --- 4. Save the Model ---
print(f"Saving trained model and tokenizer to '{SAVE_DIR}'...")
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Save complete.")

# --- 5. Evaluation & JSON Generation ---
print("Starting Inference and JSON generation...")
model.eval()
submission_results = []

with torch.no_grad():
    for batch in eval_loader:
        inputs = tokenizer(batch["prompts"], padding=True, truncation=True, max_length=1536, return_tensors="pt").to(device)
        
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        
        sub_logits = outputs["subelement_logits"].cpu()
        pres_preds = outputs["presence_preds"].cpu()
        
        for i in range(len(batch["raw_posts"])):
            post = batch["raw_posts"][i]
            timeline_id = batch["timeline_ids"][i]
            
            decoded_states = decode_predictions(sub_logits[i], pres_preds[i], threshold=0.5)
            
            pred_obj = {
                "timeline_id": timeline_id,
                "post_id": post.post_id,
                "adaptive-state": decoded_states["adaptive-state"],
                "maladaptive-state": decoded_states["maladaptive-state"]
            }
            
            if len(pred_obj["adaptive-state"]) == 1 and pred_obj["adaptive-state"]["Presence"] == 1:
                del pred_obj["adaptive-state"]
            if len(pred_obj["maladaptive-state"]) == 1 and pred_obj["maladaptive-state"]["Presence"] == 1:
                del pred_obj["maladaptive-state"]
                
            submission_results.append(pred_obj)

# Save to JSON
output_file = f"./result/task1_pred_{MODEL_NAME}_epoch{EPOCHS}.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w") as f:
    json.dump(submission_results, f, indent=4)
    
print(f"Evaluation complete! Saved {len(submission_results)} predictions to '{output_file}'.")

In [ ]:
# test.py

def test_collate(batch):
    """
    batch: List[TopKSimilarInstance]
    Builds prompts identical to training collate but skips target vectorisation.
    """
    prompts      = []
    raw_posts    = []
    timeline_ids = []

    for inst in batch:
        raw_posts.append(inst.post)
        timeline_ids.append(inst.timeline_id)

        lines = []
        for rank, (ctx_post, score) in enumerate(zip(inst.similar_posts, inst.scores), 1):
            lines.append(f"### Example {rank}  (similarity: {score:.3f})")
            lines.append(f'Post: "{ctx_post.text}"')
            lines.append("Output:")

            lines.append("  Adaptive Self-State:")
            if ctx_post.adaptive_state.subelements:
                for se in ctx_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {ctx_post.adaptive_state.presence} / 5")

            lines.append("  Maladaptive Self-State:")
            if ctx_post.maladaptive_state.subelements:
                for se in ctx_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {ctx_post.maladaptive_state.presence} / 5\n")

        lines.append("### Current Post")
        lines.append(f'Post: "{inst.text}"')
        lines.append("Output:")
        prompts.append("\n".join(lines))

    return {"prompts": prompts, "raw_posts": raw_posts, "timeline_ids": timeline_ids}




device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Hyperparameters ---
MODEL_NAME = "Qwen/Qwen2.5-7B"#"Qwen/Qwen2-1.5B" 
MODEL_DIR = ""
DATA_DIR = "./data/train"      
TEST_DIR = "./data/test"        
CACHE_DIR = "./Task_1/dataset_cache"     # Directory to store cached datasets
BATCH_SIZE = 1
THRESHOLD = 0.5
K = 5 
OUTPUT_FILE = "./task1_pred.json"
wv_model_path = "./Task_1/wiki-news-300d-1M.vec"

print("\n[1/5] Loading PostEmbedder ...")
embedder = PostEmbedder(wv_model_path)

print("\n[2/5] Loading train timelines for retrieval index ...")
os.makedirs(CACHE_DIR, exist_ok=True)
cache_path = os.path.join(CACHE_DIR, f"train_dataset_k{args.k}.pkl")

if os.path.exists(cache_path):
    print(f"  Loading cached train dataset from {cache_path} ...")
    with open(cache_path, "rb") as f:
        train_dataset = pickle.load(f)
    train_timelines = load_all_timelines(DATA_DIR)
else:
    train_timelines = load_all_timelines(DATA_DIR)
    train_index     = PostIndex(train_timelines, exclude_same_timeline=True)
    train_dataset   = TopKSimilarDataset(
        train_timelines, train_index, k=K, annotated_only=True
    )
    with open(cache_path, "wb") as f:
        pickle.dump(train_dataset, f)
    print(f"  Train dataset cached -> {cache_path}")

train_index = PostIndex(train_timelines, exclude_same_timeline=False)

print("\n[3/5] Loading and embedding test posts ...")
test_timelines = load_test_timelines(TEST_DIR, embedder)

test_dataset = TopKSimilarDataset( test_timelines, train_index, k=k, annotated_only=False)   # test posts have NO annotations)
test_loader = DataLoader( test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=test_collate)
print(f"  Test instances: {len(test_dataset)}")


print(f"\n[4/5] Loading model from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for decoder-only inference

# Load base model with 4-bit quant (same config as training)
base_model = QwenSelfStatePredictor.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["classifier"],
    ),
    device_map="auto",
)
model = base_model   # LoRA adapters were merged into model_dir by save_pretrained
model.eval()

print("\n[5/5] Running inference ...")
submission = []
total      = len(test_loader)

with torch.no_grad():
    for step, batch in enumerate(test_loader):
        inputs = tokenizer(
            batch["prompts"],
            padding=True,
            truncation=True,
            max_length=1536,
            return_tensors="pt",
        ).to(device)

        outputs    = model(input_ids=inputs["input_ids"],
                            attention_mask=inputs["attention_mask"])
        sub_logits = outputs["subelement_logits"].cpu()
        pres_preds = outputs["presence_preds"].cpu()

        for i, (post, tid) in enumerate(zip(batch["raw_posts"], batch["timeline_ids"])):
            decoded = decode_predictions(sub_logits[i], pres_preds[i], args.threshold)

            pred_obj = {
                "timeline_id":       tid,
                "post_id":           post.post_id,
                "adaptive-state":    decoded["adaptive-state"],
                "maladaptive-state": decoded["maladaptive-state"],
            }

            # Omit state entirely if presence=1 and no subelements predicted
            # (mirrors the format in the example pred.json)
            if (len(pred_obj["adaptive-state"]) == 1
                    and pred_obj["adaptive-state"]["Presence"] == 1):
                del pred_obj["adaptive-state"]

            if (len(pred_obj["maladaptive-state"]) == 1
                    and pred_obj["maladaptive-state"]["Presence"] == 1):
                del pred_obj["maladaptive-state"]

            submission.append(pred_obj)

        if (step + 1) % 10 == 0 or (step + 1) == total:
            print(f"  {step + 1}/{total} batches done", end="\r")

print()

# ── Write output ──────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(os.path.abspath(args.output_file)), exist_ok=True)
with open(args.output_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=4, ensure_ascii=False)

# Stats
has_ada = sum(1 for r in submission if "adaptive-state"    in r)
has_mal = sum(1 for r in submission if "maladaptive-state" in r)
has_both= sum(1 for r in submission if "adaptive-state" in r and "maladaptive-state" in r)
print(f"\nWrote {len(submission)} records -> {args.output_file}")
print(f"  With adaptive-state    : {has_ada}")
print(f"  With maladaptive-state : {has_mal}")
print(f"  With both states       : {has_both}")
print(f"  With neither state     : {len(submission) - has_ada - has_mal + has_both}")